# Sectra Memory Analyst: Memory Forensics Chat (UI)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danjethh/sectratek-AI-Lab/blob/main/df_llm_labs/memory_forensics_agent/memory_agent_ui.ipynb)

**Course:** AI for Cybersecurity and Digital Forensics  ·  **SectraTEK Lab**

A chat window over Volatility 3: ask about the memory image, and Sectra picks a plugin, runs it,
reads the output, and answers with the plugin and PID behind every claim. The side panel shows the
Volatility commands run and the findings recorded. In Colab you get a public link to open.

Check its answers against the [answer key](https://github.com/danjethh/sectratek-AI-Lab/blob/main/df_llm_labs/memory_forensics_agent/answer_key.md).

### 1. Install, and write the lab code

In [ ]:
%pip install -q volatility3 openai requests gradio

In [ ]:
import os
os.makedirs('case', exist_ok=True)

In [ ]:
%%writefile agentkit.py
"""
agentkit.py - the shared forensic-agent core used by the Lab 4 notebooks and chatbots.

One place defines: how to connect to a model (OpenRouter or Ollama), how to build the
synthetic Android image, the read-only tools the agent may call, the think-act-observe
loop, and a conversational wrapper for the chatbots. The chatbot .py apps import this
module; the teaching notebooks embed the same code inline so students open only one file.
"""
import json
import os
import re
import sqlite3
from pathlib import Path


# ======================================================================= model backend
def connect_model(provider="openrouter", model=None, api_key=None, base_url=None):
    """Return (client, model_id). provider is 'openrouter' or 'ollama'.

    openrouter: free hosted Qwen, needs a key from https://openrouter.ai/keys.
    ollama:     a Qwen model running locally (https://ollama.com), no key, nothing leaves the PC.
    """
    from openai import OpenAI
    provider = provider.lower()
    if provider == "ollama":
        base_url = base_url or "http://localhost:11434/v1"
        client = OpenAI(base_url=base_url, api_key="ollama")  # Ollama ignores the key
        return client, (model or "qwen3:8b")
    # default: OpenRouter
    base_url = base_url or "https://openrouter.ai/api/v1"
    api_key = api_key or os.environ.get("OPENROUTER_API_KEY", "")
    client = OpenAI(base_url=base_url, api_key=api_key)
    return client, (model or "qwen/qwen3-30b-a3b:free")


# ======================================================================= synthetic image
def build_image(root="android_image"):
    """Create a tiny, synthetic Android extraction on disk. Everything in it is fictional."""
    ROOT = Path(root)
    text_files = {
        f"{root}/build.prop":
            "# Synthetic build.prop (teaching data, fictional)\n"
            "ro.product.manufacturer=Google\nro.product.brand=google\n"
            "ro.product.model=Pixel 7a\nro.product.device=lynx\nro.product.name=lynx\n"
            "ro.build.version.release=14\nro.build.version.sdk=34\n"
            "ro.build.fingerprint=google/lynx/lynx:14/AP2A.240705.005/2024072800:user/release-keys\n",
        f"{root}/data/system/packages.list":
            "com.google.android.gm 10089 0 /data/user/0/com.google.android.gm default 0 1\n"
            "com.android.providers.contacts 10003 0 /data/user/0/com.android.providers.contacts platform 0 1\n"
            "com.android.providers.telephony 10004 0 /data/user/0/com.android.providers.telephony platform 0 1\n"
            "com.whatsapp 10137 0 /data/user/0/com.whatsapp default 0 1\n"
            "com.instagram.android 10140 0 /data/user/0/com.instagram.android default 0 1\n"
            "org.thoughtcrime.securesms 10142 0 /data/user/0/org.thoughtcrime.securesms default 0 1\n",
        f"{root}/data/system/users/0/settings_secure.xml":
            "<?xml version='1.0' encoding='utf-8' standalone='yes' ?>\n<settings version=\"241\">\n"
            "  <setting id=\"1\" name=\"android_id\" value=\"a1b2c3d4e5f60718\" package=\"android\" />\n"
            "  <setting id=\"2\" name=\"bluetooth_name\" value=\"Pixel 7a\" package=\"com.android.settings\" />\n"
            "  <setting id=\"3\" name=\"device_name\" value=\"Nadia's Pixel 7a\" package=\"com.android.settings\" />\n"
            "</settings>\n",
        f"{root}/data/misc/wifi/WifiConfigStore.xml":
            "<?xml version='1.0' encoding='utf-8' standalone='yes' ?>\n<WifiConfigStoreData>\n  <NetworkList>\n"
            "    <Network><string name=\"SSID\">&quot;HomeNet_5G&quot;</string>"
            "<long name=\"LastConnectedTime\" value=\"1721295600000\" /></Network>\n"
            "    <Network><string name=\"SSID\">&quot;BrightBeans Cafe&quot;</string>"
            "<long name=\"LastConnectedTime\" value=\"1720950000000\" /></Network>\n"
            "    <Network><string name=\"SSID\">&quot;Okafor-Office&quot;</string>"
            "<long name=\"LastConnectedTime\" value=\"1721210400000\" /></Network>\n"
            "  </NetworkList>\n</WifiConfigStoreData>\n",
        # a JSON artifact (many Android apps store prefs/state as JSON)
        f"{root}/data/data/com.whatsapp/files/me.json":
            json.dumps({"jid": "15550143788@s.whatsapp.net", "pushname": "Nadia O",
                        "registered": True, "last_backup": "2024-07-18"}, indent=2),
    }
    for rel, content in text_files.items():
        p = Path(rel)
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_text(content)

    def make_db(rel, schema, inserts):
        p = Path(rel)
        p.parent.mkdir(parents=True, exist_ok=True)
        if p.exists():
            p.unlink()
        con = sqlite3.connect(p)
        con.executescript(schema)
        for sql, params in inserts:
            con.executemany(sql, params)
        con.commit()
        con.close()

    make_db(f"{root}/data/data/com.google.android.gm/databases/accounts.db",
            "CREATE TABLE accounts(name TEXT, type TEXT, last_sync INTEGER);",
            [("INSERT INTO accounts VALUES (?,?,?)",
              [("nadia.okafor@gmail.com", "com.google", 1721295600),
               ("+15550143788", "com.whatsapp", 1721240000)])])
    make_db(f"{root}/data/data/com.android.providers.contacts/databases/contacts2.db",
            "CREATE TABLE contacts(display_name TEXT, phone TEXT, times_contacted INTEGER);",
            [("INSERT INTO contacts VALUES (?,?,?)",
              [("Marcus Webb", "+15550142210", 31), ("Priya Shah", "+15550148890", 12),
               ("Dad", "+15550149001", 54), ("BrightBeans Cafe", "+15550140077", 3),
               ("Alina K (work)", "+15550147325", 20)])])
    make_db(f"{root}/data/data/com.android.providers.telephony/databases/mmssms.db",
            "CREATE TABLE sms(address TEXT, date INTEGER, body TEXT, type INTEGER);",
            [("INSERT INTO sms VALUES (?,?,?,?)",
              [("+15550142210", 1721200000000, "Are we still on for Friday?", 1),
               ("+15550142210", 1721200600000, "Yes, 6pm at the usual place.", 2),
               ("+15550149001", 1721240500000, "Landed safe, call you tonight.", 2),
               ("+15550148890", 1721250000000, "Can you send the invoice?", 1),
               ("+15550149001", 1721299000000, "Thanks Dad, talk soon.", 2)])])
    make_db(f"{root}/data/data/com.whatsapp/databases/msgstore.db",
            "CREATE TABLE messages(chat TEXT, timestamp INTEGER, from_me INTEGER, text TEXT);",
            [("INSERT INTO messages VALUES (?,?,?,?)",
              [("Marcus Webb", 1721260000000, 0, "Did the package arrive?"),
               ("Marcus Webb", 1721260400000, 1, "Not yet, will check tomorrow."),
               ("Priya Shah", 1721280000000, 1, "Sent the files to your email.")])])
    return ROOT


# ======================================================================= the tools
# A text-ish file is one the agent can read as text. Everything else is "binary".
_TEXT_EXT = {".txt", ".xml", ".json", ".prop", ".list", ".log", ".csv", ".ini", ".conf", ".html"}


def make_tools(root="android_image", allow_python=False):
    """Build the read-only toolbox, all sealed inside `root`. Returns {name: function}."""
    ROOT = Path(root)

    def _resolve(path):
        base = ROOT.resolve()
        path = (path or ".").strip().lstrip("/")
        target = base if path in ("", ".") else (ROOT / path).resolve()
        if target != base and base not in target.parents:
            raise ValueError("path is outside the image")
        return target

    def list_dir(path="."):
        """List what is inside a folder (folders shown with a trailing '/')."""
        t = _resolve(path)
        if not t.exists():
            return f"No such path: {path}"
        if t.is_file():
            return f"{path} is a file, not a folder. Use read_file or run_sql."
        items = [e + ("/" if (t / e).is_dir() else "") for e in sorted(os.listdir(t))]
        return "\n".join(items) if items else "(empty folder)"

    def read_file(path, max_chars=1500):
        """Read a file. Pretty-prints JSON; for SQLite databases points you to run_sql;
        for other binary files shows the type and the first bytes."""
        t = _resolve(path)
        if not t.exists() or t.is_dir():
            return f"No such file: {path}"
        head = t.read_bytes()[:16]
        if head[:15] == b"SQLite format 3":
            con = sqlite3.connect(f"file:{t}?mode=ro", uri=True)
            tables = [r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type='table'")]
            con.close()
            return f"[{path} is a SQLite database. Tables: {tables}. Use run_sql with a SELECT query.]"
        ext = t.suffix.lower()
        if ext not in _TEXT_EXT and b"\x00" in head:
            return (f"[{path} is a binary file ({t.stat().st_size} bytes). First bytes (hex): "
                    f"{head.hex(' ')}. It is not readable as text.]")
        text = t.read_text(errors="replace")
        if ext == ".json":
            try:
                text = json.dumps(json.loads(text), indent=2)
            except Exception:
                pass
        return text[:max_chars] + ("" if len(text) <= max_chars else "\n...[truncated]")

    def run_sql(path, query):
        """Run ONE read-only SQL SELECT against a SQLite database and return the rows."""
        t = _resolve(path)
        if not t.exists():
            return f"No such database: {path}"
        q = (query or "").strip().rstrip(";")
        if not q.lower().startswith(("select", "pragma")):
            return "Only read-only SELECT queries are allowed."
        try:
            con = sqlite3.connect(f"file:{t}?mode=ro", uri=True)
            cur = con.execute(q)
            cols = [c[0] for c in cur.description] if cur.description else []
            rows = cur.fetchmany(50)
            con.close()
        except Exception as e:
            return f"SQL error: {e}"
        if not rows:
            return "(no rows)"
        return "\n".join([" | ".join(cols)] + [" | ".join(str(c) for c in r) for r in rows])

    def find_files(name):
        """Find files anywhere in the image whose path contains `name` (case-insensitive)."""
        base = ROOT.resolve()
        needle = (name or "").lower()
        hits = []
        for p in sorted(base.rglob("*")):
            if p.is_file() and needle in str(p.relative_to(base)).lower():
                hits.append(str(p.relative_to(base)))
        if not hits:
            return f"No files matched {name!r}."
        return "\n".join(hits[:50]) + ("" if len(hits) <= 50 else f"\n...({len(hits)} total)")

    def search_text(keyword, max_hits=20):
        """Search the text inside readable files for `keyword` (case-insensitive).
        Returns path:line for each match, so you know which file to open next."""
        base = ROOT.resolve()
        needle = (keyword or "").lower()
        out = []
        for p in sorted(base.rglob("*")):
            if not p.is_file() or p.suffix.lower() not in _TEXT_EXT:
                continue
            try:
                for i, line in enumerate(p.read_text(errors="replace").splitlines(), 1):
                    if needle in line.lower():
                        out.append(f"{p.relative_to(base)}:{i}: {line.strip()[:120]}")
                        if len(out) >= max_hits:
                            return "\n".join(out) + "\n...[more matches not shown]"
            except Exception:
                continue
        return "\n".join(out) if out else f"No text matches for {keyword!r}."

    tools = {"list_dir": list_dir, "read_file": read_file, "run_sql": run_sql,
             "find_files": find_files, "search_text": search_text}

    if allow_python:
        def run_python(code):
            """ADVANCED, OPTIONAL: run a short read-only Python snippet. `ROOT` (the image folder,
            a pathlib.Path) is available. Print what you want to see. Writing to files is blocked."""
            import io
            import contextlib
            if any(bad in code for bad in ("open(", "write", "unlink", "remove", "rmtree",
                                           "os.system", "subprocess", "Popen", "__import__")):
                return "Blocked: that snippet looks like it writes or runs external commands."
            buf = io.StringIO()
            safe = {"ROOT": ROOT, "sqlite3": sqlite3, "json": json, "Path": Path, "re": re}
            try:
                with contextlib.redirect_stdout(buf):
                    exec(code, safe)  # teaching tool: sandboxed namespace, read-only guard above
            except Exception as e:
                return f"Python error: {e}\n{buf.getvalue()}"
            out = buf.getvalue().strip()
            return out[:1500] if out else "(the snippet ran but printed nothing)"
        tools["run_python"] = run_python

    return tools


def tool_descriptions(tools):
    """One line per tool, from its docstring, for the system prompt."""
    sigs = {
        "list_dir": "list_dir(path)", "read_file": "read_file(path)",
        "run_sql": "run_sql(path, query)", "find_files": "find_files(name)",
        "search_text": "search_text(keyword)", "run_python": "run_python(code)",
    }
    lines = []
    for name, fn in tools.items():
        doc = (fn.__doc__ or "").strip().split("\n")[0]
        lines.append(f"- {sigs.get(name, name)}: {doc}")
    return "\n".join(lines)


# ======================================================================= the agent
def build_system_prompt(tools):
    return (
        "You are Sectra, a digital-forensics examiner's assistant. You are investigating an Android "
        "device extraction (a copy of the phone's files) to answer the examiner's question. You cannot "
        "see any file until you open it with a tool, and you must base every answer only on what the "
        "tools return.\n\n"
        "TOOLS (use exactly one per step):\n" + tool_descriptions(tools) + "\n\n"
        "HOW TO REPLY: send ONE JSON object and nothing else.\n"
        "To use a tool:\n"
        '  {"thought": "<one short sentence>", "action": "list_dir", "action_input": {"path": "data/data"}}\n'
        "When you can answer, reply:\n"
        '  {"thought": "<one short sentence>", "final_answer": "<your answer>", "evidence_path": "<the file you used>"}\n\n'
        "RULES: investigate step by step; use ONLY what the tools return; never invent paths, tables, "
        "or data; if a tool errors, try another path or tool; if the evidence is not there, say so "
        "plainly; always cite the exact file in evidence_path. Keep thoughts to one sentence. /no_think"
    )


def parse_action(raw):
    """Pull the JSON object out of the model's reply, tolerating <think> blocks and code fences."""
    text = re.sub(r"<think>.*?</think>", "", raw, flags=re.S)
    text = text.replace("```json", "").replace("```", "")
    start = text.find("{")
    if start == -1:
        return None
    depth = 0
    for i in range(start, len(text)):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                try:
                    return json.loads(text[start:i + 1])
                except Exception:
                    return None
    return None


def run_agent_iter(question, client, model, tools, system_prompt=None, max_steps=12, history=None):
    """Generator version of the agent, for live UIs. Yields a dict per step
    ({step, thought, action, args, observation}) and finally the answer
    ({final_answer, evidence_path}) or a give-up dict ({final_answer: None}).

    history: optional prior [(user, assistant)] turns, so the agent remembers a conversation.
    """
    system_prompt = system_prompt or build_system_prompt(tools)
    messages = [{"role": "system", "content": system_prompt}]
    for u, a in (history or []):
        messages.append({"role": "user", "content": u})
        messages.append({"role": "assistant", "content": a})
    messages.append({"role": "user", "content": f"Investigate and answer this question: {question}"})

    for step in range(1, max_steps + 1):
        raw = client.chat.completions.create(
            model=model, messages=messages, temperature=0).choices[0].message.content
        messages.append({"role": "assistant", "content": raw})
        move = parse_action(raw)
        if move is None:
            messages.append({"role": "user",
                             "content": "That was not valid JSON. Reply with ONE JSON object only."})
            continue
        if "final_answer" in move:
            yield {"final_answer": move["final_answer"], "evidence_path": move.get("evidence_path", "")}
            return
        tool, args = move.get("action"), move.get("action_input", {}) or {}
        if tool not in tools:
            observation = f"Unknown tool {tool!r}. Use one of: {', '.join(tools)}."
        else:
            try:
                observation = tools[tool](**args)
            except Exception as e:
                observation = f"Tool error: {e}"
        yield {"step": step, "thought": move.get("thought", ""),
               "action": tool, "args": args, "observation": observation}
        messages.append({"role": "user", "content": f"Observation:\n{observation}"})
    yield {"final_answer": None, "evidence_path": ""}


def run_agent(question, client, model, tools, system_prompt=None, max_steps=12,
              on_step=None, history=None):
    """Drive the model think->act->observe until it answers.

    on_step(dict): optional callback for live UIs (gets each step dict, then the final dict).
    history: optional prior [(user, assistant)] turns, so the agent remembers a conversation.
    Returns the final move dict (with 'final_answer'/'evidence_path'), or None if it gave up.
    """
    final = None
    for ev in run_agent_iter(question, client, model, tools, system_prompt, max_steps, history):
        if on_step:
            on_step(ev)
        if "final_answer" in ev:
            final = ev if ev["final_answer"] is not None else None
    return final


def answer_text(move):
    """Format a final move as a human answer line with its citation."""
    if not move:
        return "I could not reach an answer within the step limit. Try rephrasing the question."
    cite = move.get("evidence_path") or "(no file cited)"
    return f"{move['final_answer']}\n\nSource: {cite}"


In [ ]:
%%writefile memkit.py
"""
memkit.py - Sectra Memory Analyst: a memory-forensics agent over Volatility 3.

THE IDEA
--------
Volatility already gives an examiner a list of plugins (pslist, pstree, malfind, ...). Here that
plugin list IS the agent's toolset. The analyst asks a question in plain English; the language model
picks ONE plugin and its arguments; our code runs `vol -r json -f <image> <plugin>`; the model reads
the result and either runs another plugin or answers, citing the plugin and the PID it relied on.

DESIGN RULES (the same evidence discipline as the rest of the course)
---------------------------------------------------------------------
- A fixed WHITELIST of read-only Volatility plugins. The model can only choose from it; nothing is
  passed to a shell, and arguments are validated (a PID must be a number).
- Every plugin runs ONCE per image and is cached; later questions reuse the cached result.
- Output is filtered (by PID or keyword) and capped, so the model is not flooded.
- A malfind hit is a LEAD, not proof. Every claim must cite the plugin and PID behind it.
- Volatility 3's network plugins do not support Windows XP; on XP, search_memory (vadregexscan) finds
  network indicators inside process memory instead.

Reuses the shared agent loop in agentkit (connect_model, run_agent, run_agent_iter).
"""
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

import agentkit as ak

def connect_model(provider="groq", model=None, api_key=None, base_url=None):
    """Return (client, model_id). provider: 'groq' (default; free key at console.groq.com, the same
    endpoint as the agentic-pattern labs), 'openrouter' (free Qwen), or 'ollama' (local, no key)."""
    if provider.lower() == "groq":
        from openai import OpenAI
        client = OpenAI(base_url=base_url or "https://api.groq.com/openai/v1",
                        api_key=api_key or os.environ.get("GROQ_API_KEY", ""))
        return client, (model or "llama-3.3-70b-versatile")
    return ak.connect_model(provider, model, api_key, base_url)


run_agent = ak.run_agent
run_agent_iter = ak.run_agent_iter
answer_text = ak.answer_text

MAX_ROWS = 40          # rows shown to the model per call
MAX_CHARS = 3500       # hard cap on one observation

# ------------------------------------------------------------------ the public sample image
# Cridex: a classic Windows XP memory image of a banking-trojan infection, widely used to teach
# memory forensics, with publicly documented ground truth. 40 MB download, 512 MB unpacked.
SAMPLE = {
    "name": "cridex",
    "url": "https://github.com/Divinemonk/memory_forensics_with_volatility/releases/download/mem_sample/cridexmemdump.zip",
    "zip_sha256": "6453d0986352f7d5287cb90eeb6b72bc7bc16b03e964bb5767d93f85c3a483b2",
    "image": "cridex.vmem",
    "image_md5": "7494f3b77db1525c1974f5380744ae46",
}


def _sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def _md5(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def fetch_sample(dest="memory", force=False):
    """Download the public Cridex memory image, verify its hashes, and unzip it. Returns the image path."""
    dest = Path(dest)
    dest.mkdir(parents=True, exist_ok=True)
    image = dest / SAMPLE["image"]
    if image.exists() and not force:
        print(f"Already have {image} ({image.stat().st_size // (1 << 20)} MB).")
        return str(image)
    zpath = dest / "cridexmemdump.zip"
    print("Downloading the Cridex sample (about 40 MB)...")
    try:
        import requests
        with requests.get(SAMPLE["url"], stream=True, timeout=120) as r:
            r.raise_for_status()
            with open(zpath, "wb") as f:
                for chunk in r.iter_content(1 << 20):
                    f.write(chunk)
    except ImportError:
        import urllib.request
        try:
            urllib.request.urlretrieve(SAMPLE["url"], zpath)
        except Exception as e:
            raise RuntimeError(f"Download failed ({e}). Run `pip install requests` and try again.") from e
    got = _sha256(zpath)
    if got != SAMPLE["zip_sha256"]:
        raise RuntimeError(f"Download hash mismatch ({got}); the file may have changed. Not using it.")
    with zipfile.ZipFile(zpath) as z:
        z.extract(SAMPLE["image"], dest)
    zpath.unlink()
    md5 = _md5(image)
    print(f"Verified {image}: MD5 {md5} ({'matches' if md5 == SAMPLE['image_md5'] else 'DIFFERENT from'} the published sample).")
    return str(image)


# ------------------------------------------------------------------ the toolset: whitelisted plugins
# name -> spec. vol: candidate plugin names (newest first; Volatility renames some between versions).
# pid: None (not used) / "optional" / "required". keyword: None / "optional" / "required".
PLUGINS = {
    "info": dict(vol=["windows.info"], cols=["Variable", "Value"],
                 doc="OS version, build, and capture time of the memory image. A good first step."),
    "pslist": dict(vol=["windows.pslist"], pid="optional",
                   cols=["PID", "PPID", "ImageFileName", "Threads", "Handles", "CreateTime", "ExitTime"],
                   doc="Running processes from the active process list (PID, parent PID, name, start time)."),
    "pstree": dict(vol=["windows.pstree"], tree=True,
                   cols=["PID", "PPID", "ImageFileName", "CreateTime"],
                   doc="Processes as a parent/child tree: shows which process launched which."),
    "psscan": dict(vol=["windows.psscan"], pid="optional",
                   cols=["PID", "PPID", "ImageFileName", "CreateTime", "ExitTime"],
                   doc="Scans raw memory for process objects, including exited or hidden ones pslist misses."),
    "psxview": dict(vol=["windows.malware.psxview", "windows.psxview"], pid="optional",
                    doc="Cross-checks several process lists; a process missing from a view may be hidden."),
    "cmdline": dict(vol=["windows.cmdline"], pid="optional", cols=["PID", "Process", "Args"],
                    doc="The command line each process was started with."),
    "dlllist": dict(vol=["windows.dlllist"], pid="required", keyword="optional",
                    cols=["PID", "Process", "Name", "Path"],
                    doc="DLLs loaded by ONE process (needs pid; optional keyword filters by name/path)."),
    "ldrmodules": dict(vol=["windows.malware.ldrmodules", "windows.ldrmodules"], pid="required",
                       doc="Modules mapped in ONE process and whether each loader list knows them (False = possibly unlinked/hidden). Needs pid."),
    "handles": dict(vol=["windows.handles"], pid="required", keyword="optional",
                    cols=["PID", "Process", "Type", "Name"],
                    doc="Open handles (files, registry keys, mutexes...) of ONE process (needs pid; optional keyword)."),
    "malfind": dict(vol=["windows.malware.malfind", "windows.malfind"], pid="optional",
                    doc="Memory regions that look like injected code (executable, private, often with a PE 'MZ' header). A hit is a LEAD, not proof."),
    "mutantscan": dict(vol=["windows.mutantscan"], keyword="optional",
                       doc="Named mutexes in memory; malware often creates distinctive ones (optional keyword filter)."),
    "svcscan": dict(vol=["windows.svcscan"], keyword="optional",
                    doc="Windows services: name, state, and binary path (optional keyword filter)."),
    "filescan": dict(vol=["windows.filescan"], keyword="required",
                     doc="File objects in memory. Needs a keyword (e.g. '.exe', 'Temp', 'Adobe') because the list is long."),
    "hivelist": dict(vol=["windows.registry.hivelist"], cols=["Offset", "FileFullPath"],
                     doc="Registry hives loaded in memory."),
    "getsids": dict(vol=["windows.getsids"], pid="optional", cols=["PID", "Process", "SID", "Name"],
                    doc="Which user and group SIDs a process runs as."),
    "envars": dict(vol=["windows.envars"], pid="required", keyword="optional",
                   cols=["PID", "Process", "Variable", "Value"],
                   doc="Environment variables of ONE process (needs pid)."),
    "netscan": dict(vol=["windows.netscan"], pid="optional",
                    doc="Network connections and listening sockets (Windows Vista and later ONLY; on XP use search_memory)."),
}

_NOISY = {"__children", "File output", "Hexdump", "Disasm", "Notes"}
_HEXCOLS = {"Offset", "Offset(V)", "Offset(P)", "Base", "Start VPN", "End VPN", "Physical Address"}


def _vol_cmd():
    """Find the Volatility 3 command-line entry point installed with `pip install volatility3`."""
    here = Path(sys.executable).with_name("vol")
    if here.exists():
        return str(here)
    found = shutil.which("vol")
    if found:
        return found
    raise RuntimeError("Volatility 3 is not installed. Run: pip install volatility3")


def _flatten(rows, depth=0):
    """Flatten Volatility's nested __children (pstree) into rows with a depth marker."""
    out = []
    for r in rows:
        r = dict(r)
        kids = r.pop("__children", []) or []
        r["_depth"] = depth
        out.append(r)
        out += _flatten(kids, depth + 1)
    return out


def _fmt_cell(col, val):
    if val is None or val == "None":
        return "-"
    if col in _HEXCOLS and isinstance(val, int):
        return hex(val)
    s = str(val).replace("\t", " ").replace("\n", " ")
    return s if len(s) <= 120 else s[:117] + "..."


class VolRunner:
    """Runs whitelisted plugins once per image and caches the JSON rows."""

    def __init__(self, image, cache_dir=".volcache", ran=None):
        self.image = str(image)
        self.cache_dir = Path(cache_dir)
        self.cache_dir.mkdir(parents=True, exist_ok=True)
        self.mem = {}
        self.ran = ran if ran is not None else []   # audit trail: every Volatility command used (shareable)

    def _log(self, plugin, extra, label):
        entry = "vol -f <image> " + " ".join([plugin, label or " ".join(extra)]).strip()
        if entry not in self.ran:
            self.ran.append(entry)

    def _key(self, plugin, extra):
        h = hashlib.sha1(f"{self.image}|{plugin}|{extra}".encode()).hexdigest()[:16]
        return self.cache_dir / f"{plugin.replace('.', '_')}_{h}.json"

    def run(self, candidates, extra=(), label=None):
        """Run the first plugin name this Volatility version knows; return (rows, plugin_used, error).
        label: a short human-readable stand-in for long arguments in the audit trail."""
        last_err = ""
        for plugin in candidates:
            key = self._key(plugin, "|".join(extra))
            if key in self.mem:
                self._log(plugin, extra, label)
                return self.mem[key], plugin, None
            if key.exists():
                rows = json.loads(key.read_text())
                self.mem[key] = rows
                self._log(plugin, extra, label)
                return rows, plugin, None
            cmd = [_vol_cmd(), "-q", "-r", "json", "-f", self.image, plugin, *extra]
            p = subprocess.run(cmd, capture_output=True, text=True, timeout=900)
            if p.returncode == 0 and p.stdout.strip().startswith("["):
                rows = _flatten(json.loads(p.stdout))
                key.write_text(json.dumps(rows))
                self.mem[key] = rows
                self._log(plugin, extra, label)
                return rows, plugin, None
            err = (p.stderr or p.stdout or "").strip().splitlines()
            last_err = " | ".join(err[-3:]) if err else f"exit code {p.returncode}"
            if "invalid choice" not in last_err:
                break                       # a real failure, not just an unknown plugin name
        return None, candidates[0], last_err


def _table(rows, cols=None, limit=MAX_ROWS, tree=False):
    if not rows:
        return "(no rows)"
    cols = cols or [c for c in rows[0] if c not in _NOISY and not c.startswith("_")]
    lines = ["\t".join(cols)]
    for r in rows[:limit]:
        cells = [_fmt_cell(c, r.get(c)) for c in cols]
        if tree and r.get("_depth"):
            cells[0] = "." * r["_depth"] + cells[0]
        lines.append("\t".join(cells))
    if len(rows) > limit:
        lines.append(f"... {len(rows) - limit} more rows not shown. Narrow it with pid= or keyword=.")
    return "\n".join(lines)


def _malfind_view(rows):
    """malfind rows are huge (hexdump + disassembly); keep what an examiner reads first."""
    if not rows:
        return "(no suspicious regions found)"
    lines = ["PID\tProcess\tStart\tProtection\tPE header?\tfirst bytes"]
    for r in rows[:MAX_ROWS]:
        dump = (r.get("Hexdump") or "").replace("\n", " ")
        first = " ".join(dump.split()[:16])
        mz = "YES (MZ - a PE file in memory)" if "4d 5a 90 00" in dump[:200] else "no"
        lines.append(f"{r.get('PID')}\t{r.get('Process')}\t{_fmt_cell('Start VPN', r.get('Start VPN'))}\t"
                     f"{r.get('Protection')}\t{mz}\t{first}")
    if len(rows) > MAX_ROWS:
        lines.append(f"... {len(rows) - MAX_ROWS} more regions. Narrow with pid=.")
    return "\n".join(lines)


def _clip(text):
    return text if len(text) <= MAX_CHARS else text[:MAX_CHARS] + "\n...[truncated - narrow with pid= or keyword=]"


# ------------------------------------------------------------------ build the agent's tools
def make_memory_tools(image, case_dir="case", cache_dir=".volcache", findings=None, ran=None):
    """Return {tool_name: function}. One tool per whitelisted plugin, plus search_memory, the case brief,
    and the findings log. `findings` collects recorded findings; `ran` collects the Volatility commands
    used (pass your own lists to keep them across calls, e.g. one chat session)."""
    vr = VolRunner(image, cache_dir, ran=ran)
    findings = findings if findings is not None else []
    case_dir = Path(case_dir)
    tools = {}

    def make_plugin_tool(name, spec):
        def tool(pid=None, keyword=None):
            if pid in ("", "None"):
                pid = None
            if pid is not None:
                try:
                    pid = int(pid)
                except (TypeError, ValueError):
                    return f"pid must be a number, got {pid!r}."
            if spec.get("pid") == "required" and pid is None:
                return f"{name} needs a pid (one process). Find candidate PIDs with pslist or pstree first."
            if spec.get("keyword") == "required" and not keyword:
                return f"{name} needs a keyword (for example '.exe' or 'Temp') because its output is long."
            rows, used, err = vr.run(spec["vol"])
            if err:
                if name == "netscan":
                    return ("netscan failed on this image. Volatility 3's network plugins only support Windows "
                            "Vista and later; on Windows XP use search_memory(kind='ip') to find network "
                            f"indicators inside process memory. (Volatility said: {err[:200]})")
                return f"{used} failed: {err[:300]}"
            if name == "info":                 # drop the local symbol-cache path (noise, not evidence)
                rows = [r for r in rows if r.get("Variable") != "Symbols"]
            if pid is not None and spec.get("pid"):
                rows = [r for r in rows if str(r.get("PID")) == str(pid)]
            if keyword:
                k = str(keyword).lower()
                rows = [r for r in rows if k in " ".join(str(v) for v in r.values()).lower()]
            body = _malfind_view(rows) if name == "malfind" else _table(rows, spec.get("cols"), tree=spec.get("tree"))
            head = f"[{used}{' --pid ' + str(pid) if pid is not None else ''}{' | filter: ' + keyword if keyword else ''}] {len(rows)} rows\n"
            return _clip(head + body)

        args = []
        if spec.get("pid"):
            args.append("pid" + ("" if spec["pid"] == "required" else "=optional"))
        if spec.get("keyword"):
            args.append("keyword" + ("" if spec["keyword"] == "required" else "=optional"))
        tool.__doc__ = f"({', '.join(args)}) {spec['doc']}" if args else spec["doc"]
        tool.__name__ = name
        return tool

    for name, spec in PLUGINS.items():
        tools[name] = make_plugin_tool(name, spec)

    def search_memory(kind="ip", text="", pid=None):
        """(kind='ip'|'url'|'text', text=optional, pid=optional) Search inside PROCESS memory (windows.vadregexscan) for network indicators (ip = IP:port), URLs, or a literal text, and show which process holds each hit."""
        patterns = {
            "ip": r"(?:25[0-5]|2[0-4][0-9]|1?[0-9]?[0-9])(?:\.(?:25[0-5]|2[0-4][0-9]|1?[0-9]?[0-9])){3}:[0-9]{2,5}",
            "url": r"https?://[A-Za-z0-9./_?=&%-]{4,80}",
        }
        if text:
            pattern, label = re.escape(str(text)), f"text {text!r}"
        elif kind in patterns:
            pattern, label = patterns[kind], kind
        else:
            return "kind must be 'ip' or 'url', or pass text='...' to search for a literal string."
        extra = ["--pattern", pattern]
        if pid not in (None, "", "None"):
            try:
                extra += ["--pid", str(int(pid))]
            except (TypeError, ValueError):
                return f"pid must be a number, got {pid!r}."
        rows, used, err = vr.run(["windows.vadregexscan"], tuple(extra), label=f"--pattern <{label}>")
        if err:
            return f"search failed: {err[:300]}"
        seen = {}
        for r in rows:
            t = str(r.get("Text", ""))
            k = (r.get("PID"), r.get("Process"), t)
            seen[k] = seen.get(k, 0) + 1
        if not seen:
            return f"[windows.vadregexscan] no {label} found in process memory."
        lines = [f"[windows.vadregexscan pattern={label}] {len(rows)} hits, {len(seen)} distinct (PID / process / text / count):"]
        for (p, proc, t), n in sorted(seen.items(), key=lambda kv: -kv[1])[:30]:
            lines.append(f"{p}\t{proc}\t{t}\tx{n}")
        return _clip("\n".join(lines))

    tools["search_memory"] = search_memory

    def case_brief():
        """The case brief: what happened, what was collected, and what the investigator needs answered."""
        p = case_dir / "case_brief.md"
        return p.read_text() if p.exists() else "No case brief found (case/case_brief.md)."

    def record_finding(finding, evidence):
        """(finding, evidence) Write a confirmed finding to the case log, with the plugin + PID that supports it."""
        if not finding or not evidence:
            return "A finding needs both the finding text and the evidence (plugin + PID) behind it."
        findings.append({"finding": str(finding), "evidence": str(evidence)})
        return f"Recorded finding #{len(findings)}."

    def findings_report():
        """Produce the findings report from everything recorded in the case log so far."""
        lines = ["FINDINGS REPORT (memory analysis)", "=" * 34,
                 f"Image: {Path(vr.image).name}", ""]
        if not findings:
            lines.append("(No findings recorded yet. Use record_finding as you confirm each one.)")
        for i, f in enumerate(findings, 1):
            lines.append(f"{i}. {f['finding']}\n   Evidence: {f['evidence']}")
        lines += ["", "Volatility commands run:"] + [f"  {c}" for c in vr.ran] + [
            "", "Note: findings are bounded by memory evidence. A malfind hit or a string in memory is a lead; "
            "confirm with disk, network, and malware analysis before attributing intent."]
        return "\n".join(lines)

    tools["case_brief"] = case_brief
    tools["record_finding"] = record_finding
    tools["findings_report"] = findings_report
    tools["_runner"] = vr               # not a tool; exposed for notebooks (stripped from the prompt)
    return tools


def agent_tools(tools):
    """The tools the model may call (drops private helpers like _runner)."""
    return {k: v for k, v in tools.items() if not k.startswith("_")}


def tool_descriptions(tools):
    return "\n".join(f"- {n}: {(fn.__doc__ or '').strip().splitlines()[0]}" for n, fn in agent_tools(tools).items())


def build_memory_system_prompt(tools, case_name="the case"):
    return (
        f"You are Sectra, a memory-forensics analyst working {case_name}. You investigate a Windows memory "
        "image with Volatility 3. You cannot see memory directly: you learn ONLY by running the plugins below, "
        "and you must base every statement on what they return.\n\n"
        "TOOLS (each Volatility plugin is a tool; use exactly ONE per step):\n" + tool_descriptions(tools) + "\n\n"
        "HOW TO REPLY: send ONE JSON object and nothing else.\n"
        "To run a plugin:\n"
        '  {"thought": "<one short sentence>", "action": "pstree", "action_input": {}}\n'
        '  {"thought": "<one short sentence>", "action": "dlllist", "action_input": {"pid": 1640}}\n'
        "When you can answer the investigator:\n"
        '  {"thought": "<one short sentence>", "final_answer": "<answer>", "evidence_path": "<plugin + PID you relied on>"}\n\n'
        "HOW TO WORK THE CASE:\n"
        "1. Pick the plugin that answers the question most directly (process questions: pslist/pstree; code "
        "injection: malfind; what a process loaded: dlllist/ldrmodules; network on Windows XP: search_memory).\n"
        "2. Read the result, then corroborate before concluding: a suspicious finding should be supported by "
        "a second plugin (for example malfind plus pstree plus cmdline).\n"
        "3. When something is confirmed, record it with record_finding (finding + plugin + PID).\n\n"
        "RULES: cite the plugin and PID in evidence_path; a malfind hit is a LEAD, not proof (system processes "
        "like csrss.exe and winlogon.exe often show benign hits); never name a malware family or claim intent "
        "unless the plugin output supports it; say plainly what memory cannot establish; keep thoughts to one "
        "sentence and answers short. /no_think"
    )


def quick_agent(question, client, model, tools, history=None, **kw):
    prompt = build_memory_system_prompt(tools)
    return run_agent(question, client, model, agent_tools(tools), prompt, history=history, **kw)


In [ ]:
%%writefile case/case_brief.md
# Case MF-2026-001: Strange behaviour on an accounts workstation (training case)

## What happened
A clerk in the accounts office of a small firm reported that their Windows XP workstation became slow
and that their online-banking session "behaved strangely" shortly after they opened a PDF attachment.
The workstation was not shut down. IT captured a copy of its physical memory (RAM) before isolating it.

## What was collected
- One memory image: `cridex.vmem` (512 MB, captured 2012-07-22).
- No disk image, no network capture. Memory is the only evidence in this case.

## What the investigator needs answered
1. What operating system is this, and when was memory captured?
2. Which processes were running, and which process launched which?
3. Is any process unusual for this system? Why?
4. Is there evidence of code injected into a running process? Which ones?
5. Are there any network indicators (addresses the machine was talking to)? Which process holds them?
6. What can the memory evidence NOT tell us, and what would you collect next?

## Ground rules
- Every finding must cite the Volatility plugin and the PID it came from.
- A suspicious result is a lead. Corroborate it with a second plugin before you call it a finding.
- This is a training case. The memory image is the public Cridex teaching sample; the story above is a
  fictional framing for the class.


In [ ]:
%%writefile memory_app.py
"""
memory_app.py - Sectra Memory Analyst: a Gradio chat for memory forensics with Volatility 3.

Ask a question about the memory image in plain English. Sectra picks a Volatility plugin, runs it,
reads the output, and answers - citing the plugin and PID behind every claim. The right-hand panel
shows the case, the Volatility commands actually run, and the findings recorded so far.

Config via environment (optional):
  PROVIDER        groq (default) | openrouter | ollama
  MODEL           model id (default: llama-3.3-70b-versatile on Groq)
  GROQ_API_KEY    your free Groq key (or OPENROUTER_API_KEY for openrouter)
  MEMORY_IMAGE    path to the memory image. Default: memory/cridex.vmem (downloaded by the notebook)
"""
import os
import gradio as gr
import memkit as mk

PROVIDER = os.environ.get("PROVIDER", "groq")
MODEL = os.environ.get("MODEL") or None
MEMORY_IMAGE = os.environ.get("MEMORY_IMAGE", "memory/cridex.vmem")
CASE_NAME = "case MF-2026-001"

_client = None
_model = None


def get_client():
    global _client, _model
    if _client is None:
        _client, _model = mk.connect_model(PROVIDER, MODEL)
    return _client, _model


def _new_state():
    # Plain data only: Gradio copies State per session, so tools are rebuilt each turn from these lists
    # (Volatility results stay cached on disk in .volcache, so rebuilding is instant).
    return {"findings": [], "ran": []}


def _tools(state):
    return mk.make_memory_tools(MEMORY_IMAGE, case_dir="case", cache_dir=".volcache",
                                findings=state["findings"], ran=state["ran"])


def _case_md():
    p = os.path.join("case", "case_brief.md")
    return open(p).read() if os.path.exists(p) else "*No case brief found.*"


def _fmt(args):
    return ", ".join(f"{k}={v!r}" for k, v in (args or {}).items())


def _as_text(c):
    """Gradio can round-trip message content as a list of parts; coerce to a string."""
    if isinstance(c, str):
        return c
    if isinstance(c, list):
        return "".join((x.get("text") or x.get("content") or "") if isinstance(x, dict) else str(x) for x in c)
    return str(c or "")


def _pairs(history):
    out, pending = [], None
    for m in history or []:
        if m.get("role") == "user":
            pending = _as_text(m.get("content"))
        elif m.get("role") == "assistant" and pending is not None:
            out.append((pending, _as_text(m.get("content")).split("---")[-1].strip()))
            pending = None
    return out[-5:]


def _side_md(state):
    ran = state["ran"]
    f = state["findings"]
    parts = [f"**Image:** `{os.path.basename(MEMORY_IMAGE)}`"
             + ("" if os.path.exists(MEMORY_IMAGE) else "  \n:warning: *not found - run the download cell or set MEMORY_IMAGE*")]
    parts.append("### Volatility commands run\n" + ("\n".join(f"- `{c}`" for c in ran) if ran else "*None yet.*"))
    parts.append("### Findings recorded\n" + ("\n".join(f"{i}. {x['finding']}  \n   *{x['evidence']}*"
                                                      for i, x in enumerate(f, 1)) if f else "*None yet.*"))
    return "\n\n".join(parts)


def on_user(message, history):
    if not message.strip():
        return "", history
    return "", (history or []) + [{"role": "user", "content": message}]


def on_bot(history, state):
    q = _as_text(history[-1]["content"])
    try:
        client, model = get_client()
    except Exception as e:
        history.append({"role": "assistant", "content": f"Could not connect to the model: {e}"})
        yield history, _side_md(state)
        return
    all_tools = _tools(state)
    tools = mk.agent_tools(all_tools)
    prompt = mk.build_memory_system_prompt(all_tools, CASE_NAME)
    trace = "*Working the case with Volatility...*\n\n"
    history = history + [{"role": "assistant", "content": trace}]
    yield history, _side_md(state)
    final = None
    try:
        for ev in mk.run_agent_iter(q, client, model, tools, prompt, max_steps=12, history=_pairs(history[:-2])):
            if "final_answer" in ev:
                final = ev
            else:
                th = (ev.get("thought") or "").strip()
                call = f"**{ev['action']}**(`{_fmt(ev['args'])}`)"
                trace += f"{ev['step']}. _{th}_\n   {call}\n\n" if th else f"{ev['step']}. {call}\n\n"
                history[-1]["content"] = trace
                yield history, _side_md(state)
    except Exception as e:
        history[-1]["content"] = trace + f"\n\n**Model call failed:** {e}\n(Free tiers are rate-limited; try again.)"
        yield history, _side_md(state)
        return
    if final and final.get("final_answer"):
        cite = final.get("evidence_path")
        ans = final["final_answer"] + (f"\n\n`evidence: {cite}`" if cite else "")
    else:
        ans = "I could not reach a supported answer within my step limit. Try a narrower question (for example, about one PID)."
    history[-1]["content"] = trace + "\n---\n" + ans
    yield history, _side_md(state)


INTRO = (
    "# Sectra Memory Analyst\n"
    "A memory-forensics analyst working **case MF-2026-001**. Ask about the memory image in plain English; "
    "Sectra picks a **Volatility 3 plugin**, runs it, reads the output, and answers - citing the plugin and "
    "PID behind every claim. A suspicious hit is a lead until a second plugin backs it up.\n\n"
    f"*Model: `{PROVIDER}` / `{MODEL or 'default'}`  ·  Every plugin is read-only and runs once (cached).*"
)

EXAMPLES = [
    "What operating system is this, and when was memory captured?",
    "Show me the process tree. Is anything unusual?",
    "Is there evidence of injected code in any process?",
    "What did PID 1640 run, and what launched it?",
    "Are there any network indicators in memory, and which process holds them?",
    "Record what you have confirmed, then write the findings report.",
]


def build_demo():
    with gr.Blocks(title="Sectra Memory Analyst") as demo:
        state = gr.State(_new_state())
        gr.Markdown(INTRO)
        with gr.Row():
            with gr.Column(scale=2):
                chat = gr.Chatbot(height=460, show_label=False)
                msg = gr.Textbox(placeholder="Ask about the memory image...", show_label=False)
                gr.Examples(examples=EXAMPLES, inputs=msg)
            with gr.Column(scale=1):
                with gr.Accordion("Case brief", open=False):
                    gr.Markdown(_case_md())
                side = gr.Markdown(_side_md(_new_state()))
        msg.submit(on_user, [msg, chat], [msg, chat]).then(on_bot, [chat, state], [chat, side])
    return demo


def launch(**kwargs):
    opts = dict(theme=gr.themes.Soft())
    opts.update(kwargs)
    build_demo().launch(**opts)


if __name__ == "__main__":
    launch()


### 2. Get the memory image (or point MEMORY_IMAGE at your own)

In [ ]:
import memkit as mk

# Downloads the public Cridex sample (about 40 MB), checks its SHA-256, and unzips it (512 MB).
MEMORY_IMAGE = mk.fetch_sample('memory')

# Using your own memory image instead? Upload it and point to it, for example:
# MEMORY_IMAGE = '/content/my_memory.raw'
print('Image:', MEMORY_IMAGE)

### 3. Connect the model and launch the chat

The first plugin run downloads Windows symbol files (about a minute); after that, answers are quick
because every plugin result is cached.

In [ ]:
import os, getpass
PROVIDER = 'groq'   # free key at console.groq.com  |  or 'openrouter' (free Qwen) or 'ollama' (local)
if PROVIDER == 'groq' and not os.environ.get('GROQ_API_KEY'):
    os.environ['GROQ_API_KEY'] = getpass.getpass('Paste your Groq API key (hidden): ').strip()
if PROVIDER == 'openrouter' and not os.environ.get('OPENROUTER_API_KEY'):
    os.environ['OPENROUTER_API_KEY'] = getpass.getpass('Paste your OpenRouter API key (hidden): ').strip()
os.environ['PROVIDER'] = PROVIDER
os.environ['MEMORY_IMAGE'] = MEMORY_IMAGE

import memory_app
memory_app.launch(share=True)